# Spaceship Titanic — LightGBM

`01_baseline.ipynb` からの変更点は **モデルだけ** です。

| | 01 baseline | 02 LightGBM |
|---|---|---|
| 特徴量 | Cabin 分割・TotalSpend・GroupSize など | **同じ** |
| 前処理 | 中央値 / 最頻値 + One-Hot | **同じ** |
| モデル | RandomForest | **LightGBM**（勾配ブースティング） |
| 01 CV | ~0.8035 | 上回れるか確認 |
| 01 LB | 0.7978 | 提出後に比較 |

**RandomForest** は木を独立に作って多数決します。**LightGBM** は前の木の「残り（誤差）」を次の木が直していく **勾配ブースティング** です。

表形式データでは LightGBM が強いことが多いですが、Titanic（古典版）では RF の方が LB が良かった例もあります。ここでは **同じ特徴量のまま差し替えた効果** を見ます。

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42

## 1. データ読み込み・特徴量

EDA は `01_baseline.ipynb` を参照。ここでは **同じ `add_group_size` / `add_features`** をそのまま使います。

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

SPEND_COLS = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]


def add_group_size(train_df: pd.DataFrame, test_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    combined = pd.concat(
        [
            train_df.assign(_split="train"),
            test_df.assign(_split="test"),
        ],
        ignore_index=True,
    )
    combined["GroupId"] = combined["PassengerId"].str.split("_", n=1).str[0]
    combined["GroupSize"] = combined.groupby("GroupId")["GroupId"].transform("count")

    train_out = combined.loc[combined["_split"] == "train"].drop(columns=["_split"])
    test_out = combined.loc[combined["_split"] == "test"].drop(columns=["_split"])
    return train_out, test_out


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    cabin_parts = out["Cabin"].astype(str).str.split("/", expand=True)
    out["Deck"] = cabin_parts[0].replace("nan", np.nan)
    out["CabinNum"] = pd.to_numeric(cabin_parts[1], errors="coerce")
    out["Side"] = cabin_parts[2].replace("nan", np.nan)

    out["TotalSpend"] = out[SPEND_COLS].fillna(0).sum(axis=1)
    out["IsAlone"] = (out["GroupSize"] == 1).astype(int)

    return out


train_gs, test_gs = add_group_size(train, test)
train_fe = add_features(train_gs)
test_fe = add_features(test_gs)

FEATURE_COLUMNS = [
    "HomePlanet",
    "CryoSleep",
    "Destination",
    "Age",
    "VIP",
    *SPEND_COLS,
    "TotalSpend",
    "Deck",
    "CabinNum",
    "Side",
    "GroupSize",
    "IsAlone",
]

X = train_fe[FEATURE_COLUMNS]
y = train_fe["Transported"].astype(int)
X_test = test_fe[FEATURE_COLUMNS]

numeric_features = [
    "Age",
    *SPEND_COLS,
    "TotalSpend",
    "CabinNum",
    "GroupSize",
    "IsAlone",
]
categorical_features = ["HomePlanet", "CryoSleep", "Destination", "VIP", "Deck", "Side"]

print(f"train: {train.shape}, features: {len(FEATURE_COLUMNS)}")

## 2. LightGBM パイプライン

前処理（`ColumnTransformer`）は 01 と同一。`model` だけ `LGBMClassifier` に差し替えます。

主なハイパーパラメータ:

| パラメータ | 意味 |
|---|---|
| `n_estimators` | 木の本数（多いほど学習は長い） |
| `learning_rate` | 1 本あたりの更新幅（小さいほど `n_estimators` とセットで使う） |
| `num_leaves` | 1 本の木の複雑さの目安 |
| `subsample` / `colsample_bytree` | 行・列のサンプリング（過学習抑制） |

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
    ]
)

model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=20,
    random_state=RANDOM_STATE,
    verbose=-1,
    n_jobs=-1,
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model),
])

## 3. CV 評価

**StratifiedKFold（5 分割）** で accuracy を測ります。**高いほど良い** です。

01 baseline の **CV ~0.8035** と比べてください。CV が上がっても LB が下がることはあり得るので、提出後に両方見ます。

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(pipeline, X, y, cv=cv, scoring="accuracy", n_jobs=-1)

print(f"CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print(f"fold scores: {cv_scores.round(4).tolist()}")
print(f"01 baseline (参考): ~0.8035")

pipeline.fit(X, y)
train_pred = pipeline.predict(X)
print(f"train accuracy: {accuracy_score(y, train_pred):.4f}")

## 4. 提出ファイル作成

`output/submission_lgbm.csv` に保存します（01 の `submission.csv` は上書きしません）。

In [ ]:
test_pred = pipeline.predict(X_test)

submission = sample_submission.copy()
submission["Transported"] = test_pred.astype(bool)

submission_path = OUTPUT_DIR / "submission_lgbm.csv"
submission.to_csv(submission_path, index=False)

print(f"saved: {submission_path.resolve()}")
submission.head(10)

In [ ]:
# Kaggle へ提出（任意）
# !uv run kaggle competitions submit -c spaceship-titanic -f ../output/submission_lgbm.csv -m "lightgbm cabin+spend+group"